# Mean delivery time per app, with error bars

**Mean Plot with Error Bars** · a worked example from the StatNex teaching guide

**The story.** An investor asks for "the average delivery time per app, with some idea of uncertainty" for three food-delivery apps: ZipEats, FeastNow and UrbanBite. The table holds the delivery times in minutes of 50 orders from each app. A mean plot with error bars answers both.

**The question.** Are the apps' average delivery times clearly different once we allow for sampling error?

**To run it**, choose **Runtime → Run all**. Colab first warns that this notebook was not written by Google; it reads none of your data, so choose **Run anyway**. The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## Before the analysis

In [ ]:
# Each script saves its figures to the folder figures/, as it would on your computer.
# This shows each new figure under the cell that saved it.
from pathlib import Path

import matplotlib
from IPython import get_ipython
from IPython.display import Image, display

matplotlib.use("Agg")
shown = set()


def show_new_figures(*_):
    for path in sorted(Path("figures").glob("*.png"), key=lambda p: p.stat().st_mtime_ns):
        if (path, path.stat().st_mtime_ns) not in shown:
            shown.add((path, path.stat().st_mtime_ns))
            display(Image(filename=str(path)))


events = get_ipython().events
for old in [f for f in events.callbacks["post_run_cell"] if f.__name__ == "show_new_figures"]:
    events.unregister("post_run_cell", old)
events.register("post_run_cell", show_new_figures)

## The table

This cell saves the table `01_basic/mean_plot.csv` (50 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
No.,ZipEats,FeastNow,UrbanBite
1.0,16.0,32.0,36.0
2.0,24.0,26.0,41.0
3.0,28.0,31.0,38.0
4.0,19.0,33.0,48.0
5.0,29.0,25.0,33.0
6.0,26.0,37.0,42.0
7.0,22.0,30.0,42.0
8.0,30.0,28.0,42.0
9.0,22.0,29.0,34.0
10.0,26.0,35.0,46.0
11.0,28.0,33.0,38.0
12.0,17.0,27.0,53.0
13.0,29.0,34.0,41.0
14.0,26.0,26.0,45.0
15.0,23.0,23.0,35.0
16.0,23.0,30.0,46.0
17.0,23.0,30.0,47.0
18.0,22.0,33.0,33.0
19.0,26.0,35.0,44.0
20.0,12.0,29.0,31.0
21.0,26.0,29.0,32.0
22.0,14.0,42.0,44.0
23.0,29.0,33.0,42.0
24.0,20.0,32.0,39.0
25.0,23.0,32.0,41.0
26.0,20.0,41.0,38.0
27.0,27.0,29.0,46.0
28.0,23.0,36.0,38.0
29.0,28.0,34.0,36.0
30.0,19.0,33.0,31.0
31.0,27.0,30.0,36.0
32.0,27.0,32.0,38.0
33.0,29.0,38.0,32.0
34.0,24.0,35.0,46.0
35.0,26.0,41.0,45.0
36.0,18.0,31.0,38.0
37.0,25.0,37.0,44.0
38.0,58.0,37.0,34.0
39.0,17.0,33.0,32.0
40.0,31.0,31.0,29.0
41.0,27.0,40.0,45.0
42.0,18.0,39.0,40.0
43.0,24.0,35.0,37.0
44.0,29.0,37.0,42.0
45.0,25.0,35.0,43.0
46.0,24.0,35.0,36.0
47.0,25.0,31.0,39.0
48.0,23.0,33.0,40.0
49.0,20.0,37.0,46.0
50.0,26.0,33.0,44.0

## Libraries

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 120)

## The data

In [ ]:
# One column per variable. Blank and text cells are left out, so columns may differ
# in length.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
columns = ["ZipEats", "FeastNow", "UrbanBite"]
series = {c: pd.to_numeric(data[c], errors="coerce").dropna().to_numpy(float) for c in columns}

## Descriptive statistics

In [ ]:
# The SD describes the data: how far a single value typically sits from its mean.
# The standard error SE = SD / √n (SciPy's stats.sem) describes how precisely the
# mean is known: how far it would typically move in another sample of the same
# size. Variance and SD divide by n - 1.
n = np.array([x.size for x in series.values()])
means = np.array([np.mean(x) for x in series.values()])
variances = np.array([np.var(x, ddof=1) for x in series.values()])
sds = np.array([np.std(x, ddof=1) for x in series.values()])
ses = np.array([stats.sem(x) for x in series.values()])
summary = pd.DataFrame(
    [n, means, variances, sds, ses,
     [x.min() for x in series.values()], [x.max() for x in series.values()]],
    index=["n", "mean", "variance", "SD", "SE", "min", "max"], columns=list(series),
)
print(summary)

## The numbers behind the three charts

In [ ]:
band = pd.DataFrame([means + sds, means + ses, means, means - ses, means - sds],
                    index=["mean + SD", "mean + SE", "mean", "mean - SE", "mean - SD"],
                    columns=list(series))
error_se = pd.DataFrame([means, ses], index=["mean", "SE"], columns=list(series))
error_sd = pd.DataFrame([means, sds], index=["mean", "SD"], columns=list(series))
print()
print(band)
print()
print(error_se)
print()
print(error_sd)

## Mean plot

In [ ]:
# Each mean with a bar of ± one SE: how precisely the mean is known. Bars that do not
# overlap are not by themselves proof of a significant difference (two ± 1 SE bars
# can just miss each other at P of about 0.16); for a comparison, draw 95%
# confidence intervals, about the mean ± 2 SE, or run the test.
spread, bar_label = ses, "mean ± SE"
Path("figures").mkdir(exist_ok=True)
fig, ax = plt.subplots()
ax.errorbar(range(len(series)), means, yerr=spread, fmt="o", capsize=6)
ax.set_xticks(range(len(series)), list(series))
ax.set_ylabel(bar_label)
fig.savefig("figures/mean_plot.png", dpi=150)
plt.close(fig)

## What the result says

The mean (average) delivery times are 24.46 minutes for ZipEats, 32.94 for FeastNow and 39.76 for UrbanBite. The error bars here show the standard error (SE). The SE tells how far a sample mean may be from the true mean. It is small: 0.92, 0.6 and 0.76 minutes.

The means are many standard errors apart, so the bars do not come close to each other. The three averages are clearly different, even after allowing for sampling error.

The standard deviation (SD) tells a different story. It is the typical distance of one order from the mean: 6.49, 4.23 and 5.34 minutes. Bars of one SD would be much longer. So always say which bars you draw: SE for the averages, SD for single orders.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.